# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [178]:
# Step 0 - Import libraries and dataset
# ---------------------------------------
# Import the libraries needed for this section
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from scipy.stats import chi2_contingency
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Standard seed used throughout the notebook, for reproducibility
SEED = 1

In [179]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [180]:
print(spaceship.shape)

(8693, 14)


In [181]:
spaceship.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


**Check for data types**

In [182]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

In [183]:
spaceship["CryoSleep"] = spaceship["CryoSleep"].astype("boolean")
spaceship["VIP"] = spaceship["VIP"].astype("boolean")

In [184]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep       boolean
Cabin            object
Destination      object
Age             float64
VIP             boolean
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [185]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [186]:
spaceship["Age"] = spaceship["Age"].fillna(spaceship["Age"].median())

In [187]:
num_cols = ["Age", "RoomService", "FoodCourt","ShoppingMall", "Spa", "VRDeck"]

for col in num_cols:
    spaceship[col] = spaceship[col].fillna(spaceship[col].median())

In [188]:
cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]

for col in cat_cols:
    spaceship[col] = spaceship[col].fillna(spaceship[col].mode()[0])

In [189]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet        0
CryoSleep         0
Cabin           199
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            200
Transported       0
dtype: int64

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [190]:
print(spaceship["Cabin"])

0          B/0/P
1          F/0/S
2          A/0/S
3          A/0/S
4          F/1/S
          ...   
8688      A/98/P
8689    G/1499/S
8690    G/1500/S
8691     E/608/S
8692     E/608/S
Name: Cabin, Length: 8693, dtype: object


In [191]:
spaceship["Cabin"].isnull().sum()

np.int64(199)

In [192]:
spaceship["Cabin"] = spaceship["Cabin"].str[0]

In [193]:
print(spaceship["Cabin"])

0       B
1       F
2       A
3       A
4       F
       ..
8688    A
8689    G
8690    G
8691    E
8692    E
Name: Cabin, Length: 8693, dtype: object


In [194]:
spaceship["Cabin"].value_counts(dropna=False)

Cabin
F      2794
G      2559
E       876
B       779
C       747
D       478
A       256
NaN     199
T         5
Name: count, dtype: int64

In [195]:
spaceship["Cabin"] = spaceship["Cabin"].fillna("Unknown")

In [196]:
spaceship["Cabin"].value_counts()

Cabin
F          2794
G          2559
E           876
B           779
C           747
D           478
A           256
Unknown     199
T             5
Name: count, dtype: int64

- Drop PassengerId and Name

In [197]:
spaceship = spaceship.drop(["PassengerId", "Name"], axis=1)

In [198]:
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8693 non-null   object 
 1   CryoSleep     8693 non-null   boolean
 2   Cabin         8693 non-null   object 
 3   Destination   8693 non-null   object 
 4   Age           8693 non-null   float64
 5   VIP           8693 non-null   boolean
 6   RoomService   8693 non-null   float64
 7   FoodCourt     8693 non-null   float64
 8   ShoppingMall  8693 non-null   float64
 9   Spa           8693 non-null   float64
 10  VRDeck        8693 non-null   float64
 11  Transported   8693 non-null   bool   
dtypes: bool(1), boolean(2), float64(6), object(3)
memory usage: 653.8+ KB


- For non-numerical columns, do dummies.

In [199]:
categorical_cols = spaceship.select_dtypes(include="object").columns
print(categorical_cols)

Index(['HomePlanet', 'Cabin', 'Destination'], dtype='object')


In [200]:
encoder = OneHotEncoder(
    drop="first",
    sparse_output=False
)

In [201]:
encoded = encoder.fit_transform(spaceship[categorical_cols])

In [202]:
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=spaceship.index
)

In [203]:
encoded_df.head()

,HomePlanet_Europa,HomePlanet_Mars,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Cabin_Unknown,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [213]:
numerical_cols = spaceship.select_dtypes(
    include=["int64", "float64"]
).columns

In [214]:
X = pd.concat([
    spaceship[numerical_cols],
    encoded_df
], axis=1)

y = spaceship["Transported"]

**Perform Train Test Split**

In [215]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size = 0.2,
    random_state = SEED,
    stratify=y)

print(X_train.shape)
print(X_test.shape)

(6954, 18)
(1739, 18)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [216]:
# Step 3 - Feature Engineering: Feature Scaling
# ---------------------------------------------

# KNN is distance-based --> standardize with StandardScaler
scaler = StandardScaler()
scaler.set_output(transform="pandas")


# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
          Age  RoomService  FoodCourt  ShoppingMall        Spa     VRDeck  \
mean  28.9015     214.4603   449.0643      169.9198   300.6163   306.0503   
std   14.3968     631.1567  1589.9282      615.2374  1099.0262  1164.1245   

      HomePlanet_Europa  HomePlanet_Mars  Cabin_B  Cabin_C  Cabin_D  Cabin_E  \
mean             0.2469           0.2041   0.0912   0.0860   0.0555   0.1025   
std              0.4312           0.4030   0.2879   0.2804   0.2290   0.3034   

      Cabin_F  Cabin_G  Cabin_T  Cabin_Unknown  Destination_PSO J318.5-22  \
mean   0.3207   0.2923   0.0006         0.0221                     0.0912   
std    0.4668   0.4549   0.0240         0.1472                     0.2879   

      Destination_TRAPPIST-1e  
mean                   0.6989  
std                    0.4588  

AFTER scaling (X_train):
         Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  \
mean  0.0000       0.0000     0.0000       -0.0000  0.0000  0.0000   
std   1

In [217]:
# Step 4 - Model Training: KNeighborsClassifier
# ---------------------------------------------

# n_neighbors=5 + weights='distance' --> avoids ties + closer neighbors count more (more robust)
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_model.fit(X_train_scaled_df, y_train)  # Fit on train

print('Model trained!')


Model trained!


- Evaluate your model's performance. Comment it

In [220]:
# Step 5 - Results
# ----------------

# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = knn_model.predict(X_train_scaled_df)  # train
y_pred_test = knn_model.predict(X_test_scaled_df)    # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test))

Accuracy (Train): 92.31%

              precision    recall  f1-score   support

       False       0.76      0.75      0.75       863
        True       0.75      0.76      0.76       876

    accuracy                           0.75      1739
   macro avg       0.75      0.75      0.75      1739
weighted avg       0.75      0.75      0.75      1739



CONCLUSION: The KNN model shows signs of overfitting. The training accuracy is 92.31%, whereas the test accuracy is only 75%, representing a substantial gap of approximately 17 percentage points. This suggests that the model performs very well on the training observations but has considerably lower performance on unseen data. The model may therefore be capturing patterns that are specific to the training set rather than generalizable patterns. The classification report shows similar precision, recall and F1-scores for both classes, indicating that the model performs relatively evenly when predicting transported and non-transported passengers.